# Data Quality - Update Checks Notebook

Use this notebook to update existing rows in `check_registry` without reloading your full add-check list.

Identity key for updates: `(check_name, workspace_id, dataset_id)`.

In [ ]:
# Configuration
# Preferred: load shared settings from Fabric config notebook.
try:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("IPython runtime not available")
    ip.run_line_magic("run", "data_quality_config_notebook")
except Exception:
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
print(f"Schema: {FULL_SCHEMA}")

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT
    check_name, model_name, workspace_id, dataset_id,
    workspace_name, dataset_name, run_frequency, is_active
FROM {FULL_SCHEMA}.check_registry
ORDER BY check_name, model_name, workspace_id, dataset_id
""").toPandas())

## How To Specify Updates

Each tuple is:
`(check_name, workspace_id, dataset_id, model_name, workspace_name, dataset_name, dax_expression, run_frequency, is_active)`

Rules:
- First 3 fields are required identity selectors
- For update fields, use `None` to keep existing value unchanged
- `run_frequency` (when provided) must be `ONCE_PER_DAY` or `MULTIPLE_PER_DAY`

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import trim, upper

# -- EDIT BELOW ---------------------------------------------------------------
updates = [
    # Example: rename model label and update DAX while keeping names/frequency/active unchanged
    (
        "Total Sales",
        "66666666-7777-8888-9999-000000000000",
        "ffffffff-1111-2222-3333-444444444444",
        "Sales AMER v2",
        None,
        None,
        'EVALUATE ROW("value", [Total Revenue])',
        None,
        None
    )
]
# -----------------------------------------------------------------------------

if len(updates) == 0:
    raise ValueError("No updates provided. Add at least one row to the updates list.")

schema = StructType([
    StructField("check_name", StringType(), False),
    StructField("workspace_id", StringType(), False),
    StructField("dataset_id", StringType(), False),
    StructField("model_name", StringType(), True),
    StructField("workspace_name", StringType(), True),
    StructField("dataset_name", StringType(), True),
    StructField("dax_expression", StringType(), True),
    StructField("run_frequency", StringType(), True),
    StructField("is_active", BooleanType(), True),
])

u = spark.createDataFrame(updates, schema=schema)
u = (
    u.withColumn("check_name", trim("check_name"))
     .withColumn("workspace_id", trim("workspace_id"))
     .withColumn("dataset_id", trim("dataset_id"))
     .withColumn("model_name", trim("model_name"))
     .withColumn("workspace_name", trim("workspace_name"))
     .withColumn("dataset_name", trim("dataset_name"))
     .withColumn("dax_expression", trim("dax_expression"))
     .withColumn("run_frequency", upper(trim("run_frequency")))
)

bad_selector_rows = u.filter(
    "check_name IS NULL OR check_name = '' OR workspace_id IS NULL OR workspace_id = '' OR dataset_id IS NULL OR dataset_id = ''"
).toPandas()
if len(bad_selector_rows) > 0:
    raise ValueError("Update rows must include non-blank check_name/workspace_id/dataset_id.")

dupe_selectors = u.groupBy("check_name", "workspace_id", "dataset_id").count().filter("count > 1").toPandas()
if len(dupe_selectors) > 0:
    raise ValueError("Duplicate update selectors found in submission.")

bad_frequency_rows = u.filter(
    "run_frequency IS NOT NULL AND run_frequency <> '' AND run_frequency NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')"
).toPandas()
if len(bad_frequency_rows) > 0:
    raise ValueError("Invalid run_frequency in updates. Allowed: ONCE_PER_DAY, MULTIPLE_PER_DAY")

u.createOrReplaceTempView("_updates")

missing_targets = spark.sql(f"""
    SELECT u.check_name, u.workspace_id, u.dataset_id
    FROM _updates u
    LEFT JOIN {FULL_SCHEMA}.check_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    WHERE t.check_name IS NULL
""").toPandas()
if len(missing_targets) > 0:
    raise RuntimeError("One or more update selectors do not exist in check_registry.")

spark.sql(f"""
    MERGE INTO {FULL_SCHEMA}.check_registry t
    USING _updates s
      ON t.check_name = s.check_name
     AND t.workspace_id = s.workspace_id
     AND t.dataset_id = s.dataset_id
    WHEN MATCHED THEN UPDATE SET
      t.model_name = COALESCE(s.model_name, t.model_name),
      t.workspace_name = COALESCE(s.workspace_name, t.workspace_name),
      t.dataset_name = COALESCE(s.dataset_name, t.dataset_name),
      t.dax_expression = COALESCE(s.dax_expression, t.dax_expression),
      t.run_frequency = COALESCE(NULLIF(s.run_frequency, ''), t.run_frequency),
      t.is_active = COALESCE(s.is_active, t.is_active)
""")

frequency_conflicts = spark.sql(f"""
    SELECT check_name
    FROM {FULL_SCHEMA}.check_registry
    GROUP BY check_name
    HAVING COUNT(DISTINCT UPPER(TRIM(COALESCE(run_frequency, '')))) > 1
""").toPandas()
if len(frequency_conflicts) > 0:
    raise RuntimeError(
        "run_frequency consistency violation detected after update for check_name(s): "
        + ", \
check_name"].tolist())
    )

print(f"Applied updates to {len(updates)} row selector(s).")
code
python
from IPython.display import display

updated_rows = spark.sql(f"""
SELECT
    t.check_name, t.model_name, t.workspace_id, t.dataset_id,
    t.workspace_name, t.dataset_name, t.run_frequency, t.is_active
FROM {FULL_SCHEMA}.check_registry t
INNER JOIN _updates u
  ON t.check_name = u.check_name
 AND t.workspace_id = u.workspace_id
 AND t.dataset_id = u.dataset_id
ORDER BY t.check_name, t.model_name, t.workspace_id, t.dataset_id
""").toPandas()

display(updated_rows)
print(f"\nVerified {len(updated_rows)} updated row(s).")